# Wave 1 — Module 1C: Contribution Reconciliation
**Deteksi indikasi "motong iuran dari gaji karyawan tapi tidak disetor"**: bandingkan iuran yang seharusnya disetor dengan yang benar-benar disetor.

Requirement yang dicakup (`requirements.md` #3, #4, #5, #6):
- Setoran aktual **lebih kecil** dari yang seharusnya → *perlu dicek — indikasi setoran tidak sesuai*
- **Toleransi**: selisih kecil (pembulatan) diabaikan, telat bayar yang dilunasi bulan berikutnya tidak dihitung
- **Recurring check**: hanya di-flag kalau kurang setor terjadi **≥ 2 bulan berturut-turut**, bukan sekali
- Directional: kelebihan setor tidak di-flag
- Output diurutkan + kolom `reason`; hanya data level perusahaan

### Aturan per periode
```
selisih      = iuran_seharusnya - iuran_disetor
kurang setor = selisih > TOL_ABS  DAN  selisih/iuran_seharusnya > TOL_PCT
telat wajar  = kurang setor bulan t, lalu bulan t+1 kelebihan bayar >= kekurangan bulan t
```
### Status per employer
| Status | Arti |
|---|---|
| `FLAGGED` | kurang setor ≥ `MIN_CONSECUTIVE` bulan berturut-turut |
| `ONE_OFF_GAP` | pernah kurang setor, tapi tidak berturut-turut → dianggap wajar |
| `NORMAL` | setoran sesuai (dalam toleransi) |
| `INSUFFICIENT_DATA` | data setoran < `MIN_PERIODS` bulan |

### File yang dibutuhkan
`remittance_timeseries.csv` (wajib) · `ground_truth.csv` (evaluasi) · `payroll_timeseries.csv` + `headcount_timeseries.csv` (hanya kalau remittance tidak punya kolom iuran seharusnya).

**Cara pakai:** Runtime → Run all. Cek sel 3 (mapping kolom) dan sel 4 (rasio setoran/seharusnya). Output tersimpan ke Drive folder `wave 0.C`.

In [ ]:
import os, glob
from dataclasses import dataclass, asdict, replace
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", 160); pd.set_option("display.width", 220)
try:
    import google.colab  # noqa
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print("Running di Colab:", IN_COLAB)

## 1. Lokasi data & output (Google Drive)

In [ ]:
BASE_DIR  = "/content/drive/MyDrive/Healthkathon Engine"
DRIVE_DIR = f"{BASE_DIR}/Dummy Healthkathon"
OUT_DRIVE = f"{BASE_DIR}/wave 0.C"

if os.environ.get("HK_DATA_DIR"):
    DATA_DIR = Path(os.environ["HK_DATA_DIR"])
elif IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = Path(DRIVE_DIR)
    if not (DATA_DIR / "remittance_timeseries.csv").exists():
        hits = glob.glob("/content/drive/MyDrive/**/remittance_timeseries.csv", recursive=True)
        if hits:
            DATA_DIR = Path(hits[0]).parent; print("⚠ DRIVE_DIR tidak ditemukan, pakai:", DATA_DIR)
else:
    DATA_DIR = Path("data")
OUT_DIR = Path(os.environ.get("HK_OUT_DIR", OUT_DRIVE if IN_COLAB else "output_module_c"))
print("DATA_DIR =", DATA_DIR); print("OUT_DIR  =", OUT_DIR)
for f in sorted(DATA_DIR.glob("*.csv")):
    print("  ✓", f.name)
assert (DATA_DIR / "remittance_timeseries.csv").exists(), "remittance_timeseries.csv belum ada!"

## 2. Intip skema

In [ ]:
FORBIDDEN_COLS = {"nama", "name", "nama_karyawan", "employee_name", "nik", "no_kartu",
                  "no_ktp", "ktp", "alamat", "address", "tanggal_lahir", "birth_date"}
def check_privacy(df, name):
    bad = FORBIDDEN_COLS & {c.lower() for c in df.columns}
    if bad:
        raise ValueError(f"{name} berisi kolom data pribadi {bad}. Hapus dulu (requirement #6).")

RAW = {}
for name in ["remittance_timeseries", "ground_truth", "payroll_timeseries", "headcount_timeseries"]:
    p = DATA_DIR / f"{name}.csv"
    if p.exists():
        df = pd.read_csv(p)
        if name != "ground_truth":
            check_privacy(df, name)
        RAW[name] = df
for name in ["remittance_timeseries", "ground_truth"]:
    if name in RAW:
        print(f"=== {name} ({len(RAW[name]):,} baris) ===")
        print(RAW[name].head(3).to_string(), "\n")

## 3. Konfigurasi & deteksi kolom
Kalau tebakan kolom salah, isi manual di `COLUMN_MAP`.

In [ ]:
COLUMN_MAP = {
    "rm_employer": None, "rm_period": None,
    "rm_expected": None,   # iuran seharusnya (mis. expected_contribution)
    "rm_actual": None,     # iuran disetor    (mis. actual_remittance)
    "pr_avg_wage": "rata2_DPI", "hc_count": None,   # hanya untuk fallback hitung iuran seharusnya
    "gt_employer": None, "gt_label": None,
}
POSITIVE_LABELS = ["REMITTANCE", "SETOR", "IURAN", "CONTRIBUTION", "PENGGELAPAN", "EMBEZZLE"]
NEGATIVE_LABELS = ["CLEAN", "NONE", "NORMAL", "-", "NAN"]

@dataclass(frozen=True)
class ConfigC:                     # Contribution reconciliation
    TOL_PCT: float = 0.02          # selisih <= 2% dari iuran seharusnya dianggap pembulatan
    TOL_ABS: float = 10_000        # selisih <= Rp10.000 diabaikan (harus lewat KEDUANYA baru dihitung)
    MIN_CONSECUTIVE: int = 2       # flag hanya kalau kurang setor >= 2 bulan BERTURUT-TURUT
    ALLOW_CATCHUP: bool = True     # kurang bulan ini tapi dilunasi bulan depan = telat wajar, tidak dihitung
    MIN_PERIODS: int = 2           # < 2 bulan data setoran -> INSUFFICIENT_DATA
    CAP_PCT: float = 0.25          # kekurangan >= 25% -> skor ternormalisasi 1.0
    # dipakai HANYA kalau remittance tidak punya kolom "iuran seharusnya":
    CONTRIB_RATE: float = 0.05     # iuran BPJS Kesehatan PPU 5% (4% pemberi kerja + 1% pekerja)
    WAGE_CAP: float = 12_000_000   # batas atas upah perhitungan iuran

CFG_C = ConfigC()

CANDIDATES = {
    "employer": ["employer_id", "id_employer", "company_id", "perusahaan_id", "id_perusahaan",
                 "badan_usaha_id", "kode_badan_usaha", "kode_bu", "npp", "employer"],
    "period": ["period", "periode", "month", "bulan", "year_month", "yearmonth", "date", "tanggal"],
    "rm_expected": ["expected_contribution", "expected_remittance", "iuran_seharusnya", "iuran_wajib",
                    "expected_amount", "tagihan", "iuran_tagihan", "billed_amount", "expected"],
    "rm_actual": ["actual_remittance", "actual_contribution", "iuran_disetor", "iuran_dibayar", "setoran",
                  "paid_amount", "amount_paid", "remitted", "disetor", "dibayar", "actual", "paid"],
    "avg_wage": ["rata2_dpi", "rata_rata_dpi", "avg_dpi", "avg_wage", "average_wage", "rata_rata_upah"],
    "headcount": ["headcount", "active_headcount", "n_active", "jumlah_karyawan", "jumlah_peserta",
                  "n_employees", "employee_count"],
    "gt_label": ["anomaly_type", "fraud_type", "label", "injected_fraud", "fraud_label", "scenario", "jenis_fraud"],
}

def pick(df, key, override=None, required=True, exclude=()):
    if df is None:
        return None
    if override:
        if override not in df.columns:
            raise KeyError(f"Kolom '{override}' tidak ada. Kolom tersedia: {list(df.columns)}")
        return override
    lower = {c.lower(): c for c in df.columns if c not in exclude}
    for cand in CANDIDATES[key]:
        if cand in lower:
            return lower[cand]
    for cand in CANDIDATES[key]:
        if len(cand) >= 4:
            for lc, orig in lower.items():
                if cand in lc:
                    return orig
    if required:
        raise KeyError(f"Tidak bisa menebak kolom '{key}'. Isi manual di COLUMN_MAP. Kolom: {list(df.columns)}")
    return None

RM, GT = RAW["remittance_timeseries"], RAW.get("ground_truth")
M = COLUMN_MAP
COLS = dict(rm_employer=pick(RM, "employer", M["rm_employer"]), rm_period=pick(RM, "period", M["rm_period"]))
COLS["rm_expected"] = pick(RM, "rm_expected", M["rm_expected"], required=False,
                           exclude={COLS["rm_employer"], COLS["rm_period"]})
COLS["rm_actual"] = pick(RM, "rm_actual", M["rm_actual"],
                         exclude={COLS["rm_employer"], COLS["rm_period"], COLS["rm_expected"]})
if GT is not None:
    COLS.update(gt_employer=pick(GT, "employer", M["gt_employer"]), gt_label=pick(GT, "gt_label", M["gt_label"]))
print("Mapping kolom:")
for k, v in COLS.items():
    print(f"  {k:12s} -> {v}")
if not COLS["rm_expected"]:
    print("ℹ remittance tidak punya kolom iuran seharusnya -> akan dihitung dari payroll x headcount x tarif")

## 4. Siapkan tabel setoran per employer per periode

In [ ]:
num = lambda s: pd.to_numeric(s, errors="coerce")
def to_month(s):
    s2 = s.astype(str).str.strip()
    s2 = s2.where(~s2.str.fullmatch(r"\d{6}"), s2.str[:4] + "-" + s2.str[4:])
    return pd.to_datetime(s2, errors="coerce").dt.to_period("M")

def build_remittance(cfg=CFG_C):
    df = pd.DataFrame({"employer_id": RM[COLS["rm_employer"]].astype(str),
                       "period": to_month(RM[COLS["rm_period"]]), "actual": num(RM[COLS["rm_actual"]])})
    if COLS["rm_expected"]:
        df["expected"] = num(RM[COLS["rm_expected"]])
        df = df.dropna(subset=["period"]).groupby(["employer_id", "period"], as_index=False)[["expected", "actual"]].sum()
        return df.dropna(), f"kolom '{COLS['rm_expected']}'"
    # fallback: hitung dari upah dilaporkan sendiri (payroll) x jumlah karyawan x tarif
    P, H = RAW.get("payroll_timeseries"), RAW.get("headcount_timeseries")
    assert P is not None and H is not None, "Butuh payroll_timeseries & headcount_timeseries untuk menghitung iuran seharusnya"
    pw = pd.DataFrame({"employer_id": P[pick(P, "employer")].astype(str), "period": to_month(P[pick(P, "period")]),
                       "avg_wage": num(P[pick(P, "avg_wage", M["pr_avg_wage"])])})
    hc = pd.DataFrame({"employer_id": H[pick(H, "employer")].astype(str), "period": to_month(H[pick(H, "period")]),
                       "hc": num(H[pick(H, "headcount", M["hc_count"])])})
    base = pw.merge(hc, on=["employer_id", "period"])
    base["expected"] = np.minimum(base["avg_wage"], cfg.WAGE_CAP) * base["hc"] * cfg.CONTRIB_RATE
    df = df.dropna(subset=["period"]).groupby(["employer_id", "period"], as_index=False)["actual"].sum()
    df = df.merge(base[["employer_id", "period", "expected"]], on=["employer_id", "period"], how="left")
    return df.dropna(), f"dihitung: min(upah, {cfg.WAGE_CAP:,.0f}) × jumlah karyawan × {cfg.CONTRIB_RATE:.0%}"

REM, REM_SOURCE = build_remittance()
print(f"Setoran: {REM.employer_id.nunique():,} employer, {REM.period.nunique()} periode "
      f"({REM.period.min()} s/d {REM.period.max()})")
print("Iuran seharusnya dari:", REM_SOURCE)
ratio = (REM["actual"] / REM["expected"]).median()
print(f"Rasio setoran/seharusnya (median): {ratio:.3f}")
if not 0.9 <= ratio <= 1.1:
    print("⚠ Median jauh dari 1 -> kemungkinan tarif/basis iuran beda (mis. data hanya porsi pemberi kerja 4%). "
          "Sesuaikan CONTRIB_RATE.")
REM.head()

## 5. Hitung selisih, toleransi, recurring check → skor C

In [ ]:
FLAGGED, NORMAL, INSUFFICIENT = "FLAGGED", "NORMAL", "INSUFFICIENT_DATA"
rp = lambda x: ("Rp" + f"{x:,.0f}".replace(",", ".")) if pd.notna(x) else "-"

ONE_OFF = "ONE_OFF_GAP"

def module_c(rem, cfg=None):
    """rem: employer_id, period, expected, actual  ->  (tabel per employer, detail per periode)"""
    cfg = cfg or CFG_C
    d = rem.sort_values(["employer_id", "period"]).reset_index(drop=True).copy()
    d["gap"] = d["expected"] - d["actual"]                        # selisih = seharusnya - disetor
    d["gap_pct"] = np.where(d["expected"] > 0, d["gap"] / d["expected"], np.nan)
    d["over_tol"] = (d["gap"] > cfg.TOL_ABS) & (d["gap_pct"] > cfg.TOL_PCT)   # tolerance band
    d["pidx"] = d["period"].dt.year * 12 + d["period"].dt.month
    g = d.groupby("employer_id", sort=False)
    nxt_is_next_month = (g["pidx"].shift(-1) - d["pidx"]) == 1
    nxt_surplus = -g["gap"].shift(-1)                              # kelebihan bayar bulan berikutnya
    d["late_paid"] = (cfg.ALLOW_CATCHUP & d["over_tol"] & nxt_is_next_month
                      & (nxt_surplus >= d["gap"] * (1 - cfg.TOL_PCT)))
    d["is_gap"] = d["over_tol"] & ~d["late_paid"]

    # panjang run berturut-turut
    run_len = np.zeros(len(d), dtype=int)
    prev_e, prev_idx, prev_gap, cur = None, None, False, 0
    for i, (e, p, gp) in enumerate(zip(d["employer_id"], d["pidx"], d["is_gap"])):
        if gp:
            cur = cur + 1 if (e == prev_e and prev_gap and p == prev_idx + 1) else 1
        else:
            cur = 0
        run_len[i] = cur
        prev_e, prev_idx, prev_gap = e, p, gp
    d["run_len"] = run_len

    rows = []
    for eid, x in d.groupby("employer_id", sort=False):
        n_p, gaps = len(x), x[x["is_gap"]]
        longest = int(x["run_len"].max())
        r = dict(employer_id=eid, n_periods=n_p, n_gap_periods=len(gaps), longest_run=longest,
                 n_late_paid=int(x["late_paid"].sum()),
                 total_expected=float(x["expected"].sum()), total_actual=float(x["actual"].sum()),
                 total_shortfall=float(gaps["gap"].sum()),
                 median_gap_pct=float(gaps["gap_pct"].median()) if len(gaps) else 0.0,
                 run_start=None, run_end=None, first_gap_period=str(gaps["period"].min()) if len(gaps) else None)
        r["cum_shortfall_pct"] = r["total_shortfall"] / r["total_expected"] if r["total_expected"] > 0 else np.nan
        if longest > 0:
            end_i = x["run_len"].idxmax()
            r["run_end"] = str(x.loc[end_i, "period"])
            r["run_start"] = str(x.loc[end_i - longest + 1, "period"])
        if n_p < cfg.MIN_PERIODS:
            r.update(status=INSUFFICIENT, raw=np.nan)
        elif longest >= cfg.MIN_CONSECUTIVE:
            r.update(status=FLAGGED, raw=r["median_gap_pct"])
        elif len(gaps):
            r.update(status=ONE_OFF, raw=float(gaps["gap_pct"].max()))
        else:
            r.update(status=NORMAL, raw=0.0)
        rows.append(r)
    e = pd.DataFrame(rows)

    def reason(r):
        late = (f" {r.n_late_paid} kali telat setor tapi dilunasi bulan berikutnya (dianggap wajar)."
                if r.n_late_paid else "")
        if r.status == INSUFFICIENT:
            return f"Data setoran baru {r.n_periods} bulan; belum bisa dinilai."
        if r.status == FLAGGED:
            extra = (f" Ada {r.n_gap_periods - r.longest_run} bulan lain yang juga kurang setor."
                     if r.n_gap_periods > r.longest_run else "")
            return (f"Setoran iuran rata-rata {r.median_gap_pct:.0%} di bawah yang seharusnya selama "
                    f"{r.longest_run} bulan berturut-turut ({r.run_start} s/d {r.run_end}); "
                    f"total kekurangan {rp(r.total_shortfall)}.{extra}{late}")
        if r.status == ONE_OFF:
            return (f"Kurang setor pada {r.first_gap_period}, tetapi tidak berulang berturut-turut — "
                    f"dianggap wajar.{late}")
        return (f"Setoran sesuai dengan yang seharusnya (selisih dalam toleransi "
                f"{cfg.TOL_PCT:.0%} / {rp(cfg.TOL_ABS)}).{late}")
    e["reason"] = [reason(r) for r in e.itertuples()]
    return e, d

def normalize(raw, status, thr, cap):
    """Normalisasi berbasis threshold: 0 -> 0.0, threshold -> 0.5, cap -> 1.0.
    Modul yang tidak FLAGGED dibatasi <= 0.49, yang FLAGGED >= 0.5."""
    raw = pd.Series(raw, dtype=float).clip(lower=0)
    below = 0.5 * raw / thr
    above = 0.5 + 0.5 * (raw - thr) / (cap - thr)
    n = pd.Series(np.where(raw < thr, below, above), index=raw.index).clip(0, 1)
    flagged = pd.Series(status, index=raw.index).eq(FLAGGED)
    n = pd.Series(np.where(flagged, np.maximum(n, 0.5), np.minimum(n, 0.49)), index=raw.index)
    return n.where(raw.notna()).round(3)

def run_module_c(cfg=CFG_C):
    res, det = module_c(REM, cfg)
    res["score_c"] = res["raw"].round(4)
    res["score_c_norm"] = normalize(res["raw"], res["status"], cfg.TOL_PCT, cfg.CAP_PCT)
    order = {FLAGGED: 0, ONE_OFF: 1, NORMAL: 2, INSUFFICIENT: 3}
    res = (res.assign(_o=res["status"].map(order))
              .sort_values(["_o", "score_c_norm", "total_shortfall"], ascending=[True, False, False], na_position="last")
              .drop(columns=["_o", "raw"]).reset_index(drop=True))
    res.insert(0, "rank", range(1, len(res) + 1))
    return res, det

RESULT_C, DETAIL_C = run_module_c()
print(RESULT_C["status"].value_counts().to_string())
RESULT_C.head(15)[["rank", "employer_id", "status", "score_c_norm", "longest_run", "median_gap_pct", "reason"]]

## 6. Evaluasi vs `ground_truth.csv`
Apakah kasus penggelapan setoran yang di-inject ke-flag? Berapa employer bersih yang salah ke-flag?
`ONE_OFF_GAP` dihitung sebagai *tidak di-flag*.

In [ ]:
def build_truth():
    if GT is None:
        return None
    lab = GT[COLS["gt_label"]].astype(str).str.upper().str.strip()
    print("Nilai label di ground truth:", lab.value_counts().to_dict())
    t = pd.DataFrame({"employer_id": GT[COLS["gt_employer"]].astype(str), "label": lab,
                      "actual": lab.str.contains("|".join(POSITIVE_LABELS), na=False)})
    return t.groupby("employer_id", as_index=False).agg(label=("label", "first"), actual=("actual", "any"))

def evaluate(result, truth, verbose=True):
    m = result.merge(truth, on="employer_id", how="left")
    m["actual"] = m["actual"].fillna(False).astype(bool)
    ev = m[m["status"] != INSUFFICIENT]
    pred = ev["status"] == FLAGGED
    tp, fp = int((pred & ev.actual).sum()), int((pred & ~ev.actual).sum())
    fn, tn = int((~pred & ev.actual).sum()), int((~pred & ~ev.actual).sum())
    res = dict(tp=tp, fp=fp, fn=fn, tn=tn,
               precision=round(tp / (tp + fp), 3) if tp + fp else np.nan,
               recall=round(tp / (tp + fn), 3) if tp + fn else np.nan,
               false_positive_rate=round(fp / (fp + tn), 3) if fp + tn else np.nan,
               bersih_one_off_tidak_diflag=int((~ev.actual & (ev.status == ONE_OFF)).sum()),
               bersih_telat_bayar_tidak_diflag=int((~ev.actual & ~pred & (ev.n_late_paid > 0)).sum()),
               positif_di_insufficient=int(m.loc[m.status == INSUFFICIENT, "actual"].sum()))
    if verbose:
        for k, v in res.items():
            print(f"  {k:32s}: {v}")
        cols = ["employer_id", "label", "status", "longest_run", "n_gap_periods", "median_gap_pct", "reason"]
        missed, fa = ev[~pred & ev.actual], ev[pred & ~ev.actual]
        if len(missed):
            print(f"\nKasus penggelapan setoran yang TERLEWAT ({len(missed)}):")
            print(missed[cols].head(15).to_string(index=False))
        if len(fa):
            print(f"\nEmployer bersih yang SALAH ke-flag ({len(fa)}):")
            print(fa[cols].head(15).to_string(index=False))
    return res

TRUTH = build_truth()
if TRUTH is not None:
    print(f"Positif (kasus Module C): {int(TRUTH.actual.sum())} employer\n")
    EVAL_C = evaluate(RESULT_C, TRUTH)

## 7. Tuning tolerance band (sweep)
- `TOL_PCT` terlalu kecil → pembulatan wajar ikut terhitung; terlalu besar → penggelapan kecil lolos.
- `MIN_CONSECUTIVE = 1` = tanpa recurring check (untuk melihat berapa one-off yang berhasil disaring).

Pilih kombinasi, tulis di `ConfigC` (sel 3) **dan** di notebook Wave 2, lalu Run all lagi.

In [ ]:
def threshold_sweep(tol_grid=(0.005, 0.01, 0.02, 0.05, 0.10), run_grid=(1, 2, 3)):
    rows = []
    for t in tol_grid:
        for k in run_grid:
            cfg = replace(CFG_C, TOL_PCT=t, MIN_CONSECUTIVE=k)
            res, _ = run_module_c(cfg)
            e = evaluate(res, TRUTH, verbose=False)
            rows.append(dict(TOL_PCT=t, MIN_CONSECUTIVE=k,
                             **{x: e[x] for x in ("tp", "fp", "fn", "precision", "recall", "false_positive_rate")}))
    return pd.DataFrame(rows)

if TRUTH is not None:
    SWEEP = threshold_sweep()
    display(SWEEP) if "display" in globals() else print(SWEEP.to_string(index=False))

## 8. Visual sanity check
Garis abu = iuran seharusnya, biru = disetor. Titik merah = kurang setor (di luar toleransi), oranye = telat tapi dilunasi.

In [ ]:
def plot_employers(ids, detail=DETAIL_C):
    ids = list(ids)
    if not ids:
        print("Tidak ada employer untuk di-plot"); return
    fig, axes = plt.subplots(len(ids), 1, figsize=(10, 2.4 * len(ids)), squeeze=False)
    for ax, eid in zip(axes[:, 0], ids):
        d = detail[detail.employer_id == eid]; x = d["period"].astype(str)
        ax.plot(x, d["expected"], color="gray", lw=2, label="seharusnya")
        ax.plot(x, d["actual"], marker="o", color="steelblue", label="disetor")
        ax.scatter(x[d.is_gap], d.loc[d.is_gap, "actual"], color="red", s=80, zorder=3, label="kurang setor")
        ax.scatter(x[d.late_paid], d.loc[d.late_paid, "actual"], color="orange", s=80, zorder=3, label="telat, dilunasi")
        ax.set_title(eid, fontsize=9, loc="left"); ax.tick_params(axis="x", rotation=45, labelsize=7)
        ax.legend(fontsize=7, loc="lower left")
    plt.tight_layout(); plt.show()

plot_employers(RESULT_C.loc[RESULT_C.status == FLAGGED, "employer_id"].head(3))
plot_employers(RESULT_C.loc[RESULT_C.n_late_paid > 0, "employer_id"].head(1))
plot_employers(RESULT_C.loc[RESULT_C.status == ONE_OFF, "employer_id"].head(1))

## 9. Simpan output ke Google Drive (`wave 0.C`)

In [ ]:
if IN_COLAB and str(OUT_DIR).startswith("/content/drive") and not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
OUT_DIR.mkdir(parents=True, exist_ok=True)
RESULT_C.to_csv(OUT_DIR / "module_c_scores.csv", index=False)
DETAIL_C.assign(period=DETAIL_C["period"].astype(str)).to_csv(OUT_DIR / "module_c_period_detail.csv", index=False)
if TRUTH is not None:
    SWEEP.to_csv(OUT_DIR / "module_c_threshold_sweep.csv", index=False)
with open(OUT_DIR / "module_c_config.txt", "w") as f:
    f.write(str(asdict(CFG_C)) + f"\nexpected_source: {REM_SOURCE}\n")
print("Tersimpan di:", OUT_DIR)
for f in sorted(OUT_DIR.glob("module_c_*")):
    print("  ✓", f.name)